# QuadraticRangeFactor

A `QuadraticRangeFactor` expresses a range measurement as a weighted quadratic residual using an auxiliary unit direction, as used in certifiable range-aided SLAM.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/sam/doc/QuadraticRangeFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np

from gtsam.symbol_shorthand import L, T, U

## Create a lifted range factor

For a translation $t$, target $l$, measured range $r$, and auxiliary unit direction $u$, a `QuadraticRangeFactor` penalizes

$$
\frac{\nu}{2}\lVert l-t-r u\rVert^2,
$$

where `weight` is the precision $\nu$. The example uses the 3D specialization, constructs a consistent target from a `Unit3` direction, and confirms that the resulting cost is zero.

In [3]:
measured_range, weight = 1.4, 2.25
factor = gtsam.QuadraticRangeFactor3(
    T(0), L(0), U(0), measured_range, weight
)

translation = np.array([0.3, -0.2, 0.1])
direction = gtsam.Unit3(np.array([0.0, 0.0, 1.0]))
target = translation + measured_range * direction.unitVector()

values = gtsam.Values()
values.insert(T(0), translation)
values.insert(L(0), target)
values.insert(U(0), direction)

assert np.isclose(factor.error(values), 0.0)
print("range:", factor.range())
print("weight:", factor.weight())

range: 1.4
weight: 2.25


## Evaluate the quadratic cost

After perturbing the target, `error()` returns one half of the weighted squared vector residual. Computing that expression explicitly makes the meaning of `range()` and `weight()` concrete.

In [4]:
perturbed_target = target + np.array([0.1, -0.2, 0.05])
values.update(L(0), perturbed_target)
residual = (
    perturbed_target
    - translation
    - measured_range * direction.unitVector()
)
expected_cost = 0.5 * weight * float(residual @ residual)
assert np.isclose(factor.error(values), expected_cost)
print("perturbed cost:", factor.error(values))

perturbed cost: 0.059062500000000025


## When to use it

Use a `QuadraticRangeFactor` when a range-aided problem is being lifted for a certifiable quadratic solver. `QuadraticRangeFactor3` uses a `Unit3` auxiliary; `QuadraticRangeFactor2` uses the first column of a `Rot2`. For ordinary nonlinear least squares without an auxiliary direction, [`RangeFactor`](RangeFactor.ipynb) is simpler.

## Source

[`QuadraticRangeFactor.h`](../QuadraticRangeFactor.h)


## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.